<a href="https://colab.research.google.com/github/phanukon330/AI-PROJECT/blob/main/AI_Recipe_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [81]:
# 1) ติดตั้งไลบรารีที่ใช้ในโปรเจกต์
!pip -q install sentence-transformers gradio "google-genai>=2.12.1,<3"


In [82]:
# 2) เชื่อมต่อ Google Drive และเตรียมตำแหน่งเก็บไฟล์
import os
import ast
import re
import numpy as np
import pandas as pd
from google.colab import drive, userdata

drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/AI_Recipe_Chatbot"
os.makedirs(PROJECT_PATH, exist_ok=True)

RAW_DATA_PATH = "/content/Food Ingredients and Recipe Dataset with Image Name Mapping.csv"
CLEAN_DATA_PATH = f"{PROJECT_PATH}/recipes_clean.csv"
EMBEDDING_PATH = f"{PROJECT_PATH}/recipe_embeddings.npy"
PROGRESS_PATH = f"{PROJECT_PATH}/embedding_progress.txt"

if not os.path.exists(RAW_DATA_PATH):
    raise FileNotFoundError(
        "ไม่พบไฟล์ Dataset ต้นฉบับ กรุณาอัปโหลดไฟล์ "
        "Food Ingredients and Recipe Dataset with Image Name Mapping.csv "
        "ไว้ในแถบ Files ของ Colab ให้ปรากฏที่ /content/ ก่อนค่ะ"
    )

print("เตรียมโฟลเดอร์โปรเจกต์เรียบร้อย:", PROJECT_PATH)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
เตรียมโฟลเดอร์โปรเจกต์เรียบร้อย: /content/drive/MyDrive/AI_Recipe_Chatbot


In [84]:
# 3) โหลด Dataset ต้นฉบับ และตรวจสอบขนาด/คอลัมน์

df = pd.read_csv(RAW_DATA_PATH)
print("ขนาดข้อมูล:", df.shape)
print("ชื่อคอลัมน์:")
print(df.columns.tolist())
df.head()


ขนาดข้อมูล: (13501, 6)
ชื่อคอลัมน์:
['Unnamed: 0', 'Title', 'Ingredients', 'Instructions', 'Image_Name', 'Cleaned_Ingredients']


,Unnamed: 0,Title,Ingredients,Instructions,Image_Name,Cleaned_Ingredients
0,0,Miso-Butter Roast Chicken With Acorn Squash Pa...,"['1 (3½–4-lb.) whole chicken', '2¾ tsp. kosher...","Pat chicken dry with paper towels, season all ...",miso-butter-roast-chicken-acorn-squash-panzanella,"['1 (3½–4-lb.) whole chicken', '2¾ tsp. kosher..."
1,1,Crispy Salt and Pepper Potatoes,"['2 large egg whites', '1 pound new potatoes (...",Preheat oven to 400°F and line a rimmed baking...,crispy-salt-and-pepper-potatoes-dan-kluger,"['2 large egg whites', '1 pound new potatoes (..."
2,2,Thanksgiving Mac and Cheese,"['1 cup evaporated milk', '1 cup whole milk', ...",Place a rack in middle of oven; preheat to 400...,thanksgiving-mac-and-cheese-erick-williams,"['1 cup evaporated milk', '1 cup whole milk', ..."
3,3,Italian Sausage and Bread Stuffing,"['1 (¾- to 1-pound) round Italian loaf, cut in...",Preheat oven to 350°F with rack in middle. Gen...,italian-sausage-and-bread-stuffing-240559,"['1 (¾- to 1-pound) round Italian loaf, cut in..."
4,4,Newton's Law,"['1 teaspoon dark brown sugar', '1 teaspoon ho...",Stir together brown sugar and hot water in a c...,newtons-law-apple-bourbon-cocktail,"['1 teaspoon dark brown sugar', '1 teaspoon ho..."


In [85]:
# 4) สำรวจ Missing Values และข้อมูลซ้ำ
print("=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Duplicate Rows ===")
print("จำนวนแถวที่ซ้ำ:", df.duplicated().sum())

print("\n=== Duplicate Titles ===")
print("จำนวนชื่อเมนูที่ซ้ำ:", df["Title"].duplicated().sum())
print("จำนวนชื่อเมนูที่ไม่ซ้ำ:", df["Title"].nunique())


=== Missing Values ===
Unnamed: 0             0
Title                  5
Ingredients            0
Instructions           8
Image_Name             0
Cleaned_Ingredients    0
dtype: int64

=== Duplicate Rows ===
จำนวนแถวที่ซ้ำ: 0

=== Duplicate Titles ===
จำนวนชื่อเมนูที่ซ้ำ: 195
จำนวนชื่อเมนูที่ไม่ซ้ำ: 13305


In [86]:
# 5) ตรวจสอบแถวที่ชื่อเมนูหรือวิธีทำหายไป
missing_data = df[df["Title"].isna() | df["Instructions"].isna()]
print("จำนวนแถวที่ชื่อเมนูหรือวิธีทำหายไป:", len(missing_data))
missing_data[["Title", "Ingredients", "Instructions"]].head(20)


จำนวนแถวที่ชื่อเมนูหรือวิธีทำหายไป: 8


,Title,Ingredients,Instructions
4293,Broccolini-Cheddar Gratin with Rye Breadcrumbs,"['1/4 loaf seeded rye bread, torn into 1"" piec...",NaN
9636,Smoked Salmon with Egg Salad and Green beans,[],NaN
10356,Royal Icing,"['Using electric mixer, beat 3 1/4 cups powder...",NaN
11221,NaN,[],NaN
12373,NaN,[],NaN
12378,NaN,[],NaN
12818,NaN,[],NaN
12829,NaN,[],NaN


In [87]:
# 6) ตรวจสอบชื่อเมนูที่ซ้ำ

duplicate_titles = df["Title"].dropna().value_counts()
duplicate_titles = duplicate_titles[duplicate_titles > 1]
print("จำนวนชื่อเมนูที่ปรากฏมากกว่า 1 ครั้ง:", len(duplicate_titles))
duplicate_titles.head(20)


จำนวนชื่อเมนูที่ปรากฏมากกว่า 1 ครั้ง: 162


,count
Title,
French 75,5
Potato Latkes,5
Sazerac,4
Pickled Red Onions,4
Chopped Salad,4
Manhattan,4
Daiquiri,3
Preserved Lemons,3
Borscht,3


In [88]:
# 7) ทำความสะอาดข้อมูลตามขั้นตอนใน Notebook เดิม
# เก็บเฉพาะสูตรที่มีชื่อเมนูและวิธีทำ
required_columns = ["Title", "Ingredients", "Instructions", "Cleaned_Ingredients"]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Dataset ขาดคอลัมน์ที่จำเป็น: {missing_columns}")

df_clean = df.dropna(subset=["Title", "Instructions", "Ingredients", "Cleaned_Ingredients"]).copy()

# ตัดช่องว่างรอบชื่อเมนู
df_clean["Title"] = df_clean["Title"].astype(str).str.strip()

# ตัดสูตรที่ไม่มีวัตถุดิบ หรือมีค่าเป็น []
df_clean["Ingredients"] = df_clean["Ingredients"].astype(str)
df_clean = df_clean[df_clean["Ingredients"].str.strip() != "[]"].copy()
df_clean = df_clean[df_clean["Title"] != ""].copy()

print("จำนวนข้อมูลก่อนทำความสะอาด:", len(df))
print("จำนวนข้อมูลหลังทำความสะอาด:", len(df_clean))
print("จำนวนแถวที่ถูกตัดออก:", len(df) - len(df_clean))


จำนวนข้อมูลก่อนทำความสะอาด: 13501
จำนวนข้อมูลหลังทำความสะอาด: 13487
จำนวนแถวที่ถูกตัดออก: 14


In [89]:
# 8) ตรวจสอบผลหลังทำความสะอาด และลบคอลัมน์ลำดับเดิมถ้ามี
print("=== Missing Values หลังทำความสะอาด ===")
print(df_clean.isnull().sum())
print("\nจำนวนสูตรที่ไม่มีวัตถุดิบ:", (df_clean["Ingredients"].str.strip() == "[]").sum())
print("จำนวนชื่อเมนูซ้ำ:", df_clean["Title"].duplicated().sum())

# Unnamed: 0 เป็นคอลัมน์ลำดับเดิม ไม่ใช่ข้อมูลสูตรอาหาร
if "Unnamed: 0" in df_clean.columns:
    df_clean = df_clean.drop(columns=["Unnamed: 0"])

print("\nขนาด Dataset หลังทำความสะอาด:", df_clean.shape)
print("คอลัมน์ที่เหลือ:", df_clean.columns.tolist())
df_clean.head()


=== Missing Values หลังทำความสะอาด ===
Unnamed: 0             0
Title                  0
Ingredients            0
Instructions           0
Image_Name             0
Cleaned_Ingredients    0
dtype: int64

จำนวนสูตรที่ไม่มีวัตถุดิบ: 0
จำนวนชื่อเมนูซ้ำ: 191

ขนาด Dataset หลังทำความสะอาด: (13487, 5)
คอลัมน์ที่เหลือ: ['Title', 'Ingredients', 'Instructions', 'Image_Name', 'Cleaned_Ingredients']


,Title,Ingredients,Instructions,Image_Name,Cleaned_Ingredients
0,Miso-Butter Roast Chicken With Acorn Squash Pa...,"['1 (3½–4-lb.) whole chicken', '2¾ tsp. kosher...","Pat chicken dry with paper towels, season all ...",miso-butter-roast-chicken-acorn-squash-panzanella,"['1 (3½–4-lb.) whole chicken', '2¾ tsp. kosher..."
1,Crispy Salt and Pepper Potatoes,"['2 large egg whites', '1 pound new potatoes (...",Preheat oven to 400°F and line a rimmed baking...,crispy-salt-and-pepper-potatoes-dan-kluger,"['2 large egg whites', '1 pound new potatoes (..."
2,Thanksgiving Mac and Cheese,"['1 cup evaporated milk', '1 cup whole milk', ...",Place a rack in middle of oven; preheat to 400...,thanksgiving-mac-and-cheese-erick-williams,"['1 cup evaporated milk', '1 cup whole milk', ..."
3,Italian Sausage and Bread Stuffing,"['1 (¾- to 1-pound) round Italian loaf, cut in...",Preheat oven to 350°F with rack in middle. Gen...,italian-sausage-and-bread-stuffing-240559,"['1 (¾- to 1-pound) round Italian loaf, cut in..."
4,Newton's Law,"['1 teaspoon dark brown sugar', '1 teaspoon ho...",Stir together brown sugar and hot water in a c...,newtons-law-apple-bourbon-cocktail,"['1 teaspoon dark brown sugar', '1 teaspoon ho..."


In [90]:
# 9) ตรวจสอบ Ingredients และ Cleaned_Ingredients
# เปรียบเทียบข้อมูลตัวอย่างและจำนวนสูตรที่มีรายการวัตถุดิบต่างกัน

sample_ingredients = ast.literal_eval(df_clean.iloc[0]["Ingredients"])
sample_cleaned = ast.literal_eval(df_clean.iloc[0]["Cleaned_Ingredients"])
print("ตัวอย่างเมนู:", df_clean.iloc[0]["Title"])
print("Ingredients:", sample_ingredients)
print("Cleaned_Ingredients:", sample_cleaned)

changed_count = 0
for _, row in df_clean.iterrows():
    original = ast.literal_eval(row["Ingredients"])
    cleaned = ast.literal_eval(row["Cleaned_Ingredients"])
    if original != cleaned:
        changed_count += 1

print("\nจำนวนสูตรทั้งหมด:", len(df_clean))
print("จำนวนสูตรที่วัตถุดิบแตกต่างกัน:", changed_count)
print("จำนวนสูตรที่วัตถุดิบเหมือนกัน:", len(df_clean) - changed_count)


ตัวอย่างเมนู: Miso-Butter Roast Chicken With Acorn Squash Panzanella
Ingredients: ['1 (3½–4-lb.) whole chicken', '2¾ tsp. kosher salt, divided, plus more', '2 small acorn squash (about 3 lb. total)', '2 Tbsp. finely chopped sage', '1 Tbsp. finely chopped rosemary', '6 Tbsp. unsalted butter, melted, plus 3 Tbsp. room temperature', '¼ tsp. ground allspice', 'Pinch of crushed red pepper flakes', 'Freshly ground black pepper', '⅓ loaf good-quality sturdy white bread, torn into 1" pieces (about 2½ cups)', '2 medium apples (such as Gala or Pink Lady; about 14 oz. total), cored, cut into 1" pieces', '2 Tbsp. extra-virgin olive oil', '½ small red onion, thinly sliced', '3 Tbsp. apple cider vinegar', '1 Tbsp. white miso', '¼ cup all-purpose flour', '2 Tbsp. unsalted butter, room temperature', '¼ cup dry white wine', '2 cups unsalted chicken broth', '2 tsp. white miso', 'Kosher salt, freshly ground pepper']
Cleaned_Ingredients: ['1 (3½–4-lb.) whole chicken', '2¾ tsp. kosher salt, divided, plus m

In [91]:
# 10) เตรียมข้อมูลสำหรับระบบค้นหาและ RAG
# ใช้ Ingredients ต้นฉบับตามแนวทางใน Notebook เดิม

def parse_ingredient_list(value):
    if isinstance(value, list):
        return value
    parsed = ast.literal_eval(str(value))
    if not isinstance(parsed, list):
        raise ValueError("รูปแบบ Ingredients ต้องเป็นรายการ (list)")
    return parsed

df_clean["Ingredients_List"] = df_clean["Ingredients"].apply(parse_ingredient_list)
df_clean["Ingredients_Text"] = df_clean["Ingredients_List"].apply(
    lambda items: ", ".join(str(item) for item in items)
)
df_clean["Search_Text"] = (
    "Recipe: " + df_clean["Title"].astype(str) +
    ". Ingredients: " + df_clean["Ingredients_Text"].astype(str)
)

print("จำนวนสูตรอาหาร:", len(df_clean))
print("Search_Text ว่าง:", df_clean["Search_Text"].isna().sum())
print("Title ว่าง:", df_clean["Title"].isna().sum())
display(df_clean[["Title", "Ingredients_Text", "Search_Text", "Instructions"]].head(5))


จำนวนสูตรอาหาร: 13487
Search_Text ว่าง: 0
Title ว่าง: 0


,Title,Ingredients_Text,Search_Text,Instructions
0,Miso-Butter Roast Chicken With Acorn Squash Pa...,"1 (3½–4-lb.) whole chicken, 2¾ tsp. kosher sal...",Recipe: Miso-Butter Roast Chicken With Acorn S...,"Pat chicken dry with paper towels, season all ..."
1,Crispy Salt and Pepper Potatoes,"2 large egg whites, 1 pound new potatoes (abou...",Recipe: Crispy Salt and Pepper Potatoes. Ingre...,Preheat oven to 400°F and line a rimmed baking...
2,Thanksgiving Mac and Cheese,"1 cup evaporated milk, 1 cup whole milk, 1 tsp...",Recipe: Thanksgiving Mac and Cheese. Ingredien...,Place a rack in middle of oven; preheat to 400...
3,Italian Sausage and Bread Stuffing,"1 (¾- to 1-pound) round Italian loaf, cut into...",Recipe: Italian Sausage and Bread Stuffing. In...,Preheat oven to 350°F with rack in middle. Gen...
4,Newton's Law,"1 teaspoon dark brown sugar, 1 teaspoon hot wa...",Recipe: Newton's Law. Ingredients: 1 teaspoon ...,Stir together brown sugar and hot water in a c...


In [92]:
# 11) บันทึก Dataset ที่ทำความสะอาดแล้วลง Google Drive

df_clean.to_csv(CLEAN_DATA_PATH, index=False)
print("บันทึก Dataset สำเร็จ:", CLEAN_DATA_PATH)
print("จำนวนสูตร:", len(df_clean))


บันทึก Dataset สำเร็จ: /content/drive/MyDrive/AI_Recipe_Chatbot/recipes_clean.csv
จำนวนสูตร: 13487


In [93]:
# 12) โหลดโมเดล SentenceTransformer สำหรับสร้าง Embeddings
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_NAME)
print("โหลดโมเดล Embedding สำเร็จ!")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

โหลดโมเดล Embedding สำเร็จ!


In [94]:
# 13) ทดลองแปลงข้อความสูตรอาหาร 1 รายการเป็น Vector
sample_text = df_clean.iloc[0]["Search_Text"]
sample_vector = model.encode(sample_text, normalize_embeddings=True)
print("ชื่อเมนู:", df_clean.iloc[0]["Title"])
print("ขนาด Vector:", sample_vector.shape)
print("ตัวเลข 10 ตัวแรก:", sample_vector[:10])


ชื่อเมนู: Miso-Butter Roast Chicken With Acorn Squash Panzanella
ขนาด Vector: (384,)
ตัวเลข 10 ตัวแรก: [-0.01875482  0.00538176  0.01233841 -0.02474887  0.02433769 -0.08697885
  0.02206316 -0.03843191 -0.02916362 -0.06205506]


In [95]:
# 14) สร้าง Embeddings ทั้ง Dataset และบันทึกแบบแบ่ง Batch
# หากเคยเริ่มทำแล้วแต่ Runtime หลุด เซลล์นี้จะทำต่อจากตำแหน่งที่บันทึกไว้

texts = df_clean["Search_Text"].tolist()
batch_size = 32
total = len(texts)
vector_size = model.get_sentence_embedding_dimension()

can_resume = os.path.exists(EMBEDDING_PATH) and os.path.exists(PROGRESS_PATH)
if can_resume:
    existing = np.load(EMBEDDING_PATH, mmap_mode="r+")
    saved_progress = int(open(PROGRESS_PATH).read().strip() or "0")
    if existing.shape != (total, vector_size) or saved_progress < 0 or saved_progress > total:
        print("ไฟล์ Embeddings เดิมไม่ตรงกับ Dataset ปัจจุบัน จะเริ่มสร้างใหม่")
        del existing
        can_resume = False

if not can_resume:
    embeddings_file = np.lib.format.open_memmap(
        EMBEDDING_PATH, mode="w+", dtype="float32", shape=(total, vector_size)
    )
    start = 0
    with open(PROGRESS_PATH, "w") as f:
        f.write("0")
else:
    embeddings_file = np.load(EMBEDDING_PATH, mmap_mode="r+")
    start = saved_progress
    print("พบงานเดิม ทำต่อจากรายการ:", start)

for i in range(start, total, batch_size):
    end = min(i + batch_size, total)
    batch_vectors = model.encode(
        texts[i:end], batch_size=batch_size,
        normalize_embeddings=True, convert_to_numpy=True,
        show_progress_bar=False
    )
    embeddings_file[i:end] = batch_vectors
    embeddings_file.flush()
    with open(PROGRESS_PATH, "w") as f:
        f.write(str(end))
    print(f"บันทึกแล้ว {end}/{total} สูตร")

# โหลด Embeddings ที่สร้างเสร็จแล้ว
embeddings = np.load(EMBEDDING_PATH)
if os.path.exists(PROGRESS_PATH):
    os.remove(PROGRESS_PATH)

print("\nสร้าง Embeddings สำเร็จ!")
print("จำนวนสูตร:", len(embeddings))
print("ขนาด Embeddings:", embeddings.shape)
print("บันทึกไว้ที่:", EMBEDDING_PATH)


/tmp/ipykernel_504/1428454569.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  vector_size = model.get_sentence_embedding_dimension()


บันทึกแล้ว 32/13487 สูตร
บันทึกแล้ว 64/13487 สูตร
บันทึกแล้ว 96/13487 สูตร
บันทึกแล้ว 128/13487 สูตร
บันทึกแล้ว 160/13487 สูตร
บันทึกแล้ว 192/13487 สูตร
บันทึกแล้ว 224/13487 สูตร
บันทึกแล้ว 256/13487 สูตร
บันทึกแล้ว 288/13487 สูตร
บันทึกแล้ว 320/13487 สูตร
บันทึกแล้ว 352/13487 สูตร
บันทึกแล้ว 384/13487 สูตร
บันทึกแล้ว 416/13487 สูตร
บันทึกแล้ว 448/13487 สูตร
บันทึกแล้ว 480/13487 สูตร
บันทึกแล้ว 512/13487 สูตร
บันทึกแล้ว 544/13487 สูตร
บันทึกแล้ว 576/13487 สูตร
บันทึกแล้ว 608/13487 สูตร
บันทึกแล้ว 640/13487 สูตร
บันทึกแล้ว 672/13487 สูตร
บันทึกแล้ว 704/13487 สูตร
บันทึกแล้ว 736/13487 สูตร
บันทึกแล้ว 768/13487 สูตร
บันทึกแล้ว 800/13487 สูตร
บันทึกแล้ว 832/13487 สูตร
บันทึกแล้ว 864/13487 สูตร
บันทึกแล้ว 896/13487 สูตร
บันทึกแล้ว 928/13487 สูตร
บันทึกแล้ว 960/13487 สูตร
บันทึกแล้ว 992/13487 สูตร
บันทึกแล้ว 1024/13487 สูตร
บันทึกแล้ว 1056/13487 สูตร
บันทึกแล้ว 1088/13487 สูตร
บันทึกแล้ว 1120/13487 สูตร
บันทึกแล้ว 1152/13487 สูตร
บันทึกแล้ว 1184/13487 สูตร
บันทึกแล้ว 1216/13487 สูตร
บันทึกแล

In [96]:
# 15) ตรวจสอบว่า Dataset และ Embeddings มีจำนวนแถวตรงกัน
assert len(df_clean) == embeddings.shape[0], "จำนวนสูตรกับ Embeddings ไม่ตรงกัน"
assert embeddings.shape[1] == model.get_sentence_embedding_dimension(), "ขนาด Vector ไม่ตรงกับโมเดล"
print("ตรวจสอบผ่าน: Dataset และ Embeddings พร้อมใช้งาน")


ตรวจสอบผ่าน: Dataset และ Embeddings พร้อมใช้งาน


/tmp/ipykernel_504/2403084285.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  assert embeddings.shape[1] == model.get_sentence_embedding_dimension(), "ขนาด Vector ไม่ตรงกับโมเดล"


In [97]:

# 16) ระบบค้นหาสูตรอาหารด้วย Embedding + ตรวจสอบวัตถุดิบ

# คำเรียกและรูปแบบอื่นของวัตถุดิบ
ingredient_variants = {
    "chicken": ["chicken", "chickens"],
    "egg": ["egg", "eggs"],
    "garlic": ["garlic"],
    "rice": ["rice"],
    "onion": ["onion", "onions"],
    "tomato": ["tomato", "tomatoes"],
    "potato": ["potato", "potatoes"],
    "shrimp": ["shrimp", "shrimps", "prawn", "prawns"],
    "pork": ["pork"],
    "beef": ["beef"],
    "fish": ["fish", "fishes"],
    "salmon": ["salmon"],
    "tuna": ["tuna"],
    "crab": ["crab", "crabs"],
    "squid": ["squid", "squids", "calamari"],
    "carrot": ["carrot", "carrots"],
    "mushroom": ["mushroom", "mushrooms"],
    "broccoli": ["broccoli"],
    "cabbage": ["cabbage"],
    "cucumber": ["cucumber", "cucumbers"],
    "ginger": ["ginger"],
    "chili": ["chili", "chilli", "chiles", "chilies"],
    "lemon": ["lemon", "lemons"],
    "lime": ["lime", "limes"],
    "milk": ["milk"],
    "cheese": ["cheese"],
    "butter": ["butter"],
    "salt": ["salt"],
    "sugar": ["sugar"],
    "basil": ["basil"],
    "cilantro": ["cilantro", "coriander"],
    "spinach": ["spinach"],
    "corn": ["corn"],
    "bell pepper": ["bell pepper", "bell peppers"],
    "green onion": ["green onion", "green onions", "scallion", "scallions"],
}


def normalize_ingredient_name(item):
    """แปลงชื่อวัตถุดิบให้เป็นชื่อมาตรฐาน"""
    item = str(item).strip().lower()

    for canonical, variants in ingredient_variants.items():
        if item in variants:
            return canonical

    return item


def ingredient_list(value):
    """รองรับทั้งข้อมูลวัตถุดิบที่เป็น List และ String"""
    if isinstance(value, list):
        return [str(item).strip() for item in value if str(item).strip()]

    if isinstance(value, str):
        return [
            item.strip()
            for item in value.split(",")
            if item.strip()
        ]

    return []


def is_ingredient_match(keyword, item):
    """ตรวจสอบว่าวัตถุดิบในสูตรตรงกับวัตถุดิบที่ผู้ใช้มีหรือไม่"""
    item_lower = str(item).lower()

    # ไม่ถือว่า chicken stock คือเนื้อไก่
    if keyword == "chicken" and re.search(
        r"\bchicken\s+(stock|broth|bouillon)\b",
        item_lower
    ):
        return False

    variants = ingredient_variants.get(keyword, [keyword])

    for word in variants:
        pattern = r"(?<![a-z])" + re.escape(word) + r"(?![a-z])"

        if re.search(pattern, item_lower):
            return True

    return False


def search_recipes(user_ingredients, top_k=5):

    # ตรวจสอบข้อมูลที่ผู้ใช้กรอก
    if not isinstance(user_ingredients, str) or not user_ingredients.strip():
        return pd.DataFrame()

    # แยกและปรับชื่อวัตถุดิบ
    ingredients = [
        normalize_ingredient_name(item)
        for item in user_ingredients.split(",")
        if item.strip()
    ]

    # ลบรายการที่ซ้ำกัน
    ingredients = list(dict.fromkeys(ingredients))

    if not ingredients:
        return pd.DataFrame()

    # แปลงวัตถุดิบเป็น Embedding
    query_vector = model.encode(
        ", ".join(ingredients),
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    # คำนวณ Similarity กับสูตรอาหารทั้งหมด
    similarities = embeddings @ query_vector

    # เลือกสูตรที่มีคะแนนใกล้เคียงสูงสุด 200 รายการ
    candidate_k = min(200, len(df_clean))

    top_indices = np.argpartition(
        similarities, -candidate_k
    )[-candidate_k:]

    top_indices = top_indices[
        np.argsort(similarities[top_indices])[::-1]
    ]

    candidates = df_clean.iloc[top_indices].copy()
    candidates["Similarity"] = similarities[top_indices]

    # ตรวจสอบวัตถุดิบที่ผู้ใช้มีและตรงกับสูตร
    def find_matched(recipe_ingredients):
        recipe_items = ingredient_list(recipe_ingredients)

        return [
            keyword
            for keyword in ingredients
            if any(
                is_ingredient_match(keyword, item)
                for item in recipe_items
            )
        ]

    candidates["Matched_Ingredients"] = (
        candidates["Ingredients_List"].apply(find_matched)
    )

    candidates["Matched_Count"] = (
        candidates["Matched_Ingredients"].apply(len)
    )

    # ตรวจสอบวัตถุดิบที่ยังไม่มี
    def find_missing(recipe_ingredients, matched):
        recipe_items = ingredient_list(recipe_ingredients)
        missing_items = []

        for item in recipe_items:

            # ถ้ามีวัตถุดิบนี้อยู่แล้ว ไม่ต้องแสดงซ้ำ
            already_have = any(
                is_ingredient_match(keyword, item)
                for keyword in matched
            )

            if not already_have:
                missing_items.append(item)

        return missing_items

    candidates["Missing_Ingredients"] = candidates.apply(
        lambda row: find_missing(
            row["Ingredients_List"],
            row["Matched_Ingredients"]
        ),
        axis=1
    )

    # เรียงตามจำนวนวัตถุดิบที่ตรงกันและคะแนน Similarity
    results = candidates.sort_values(
        by=["Matched_Count", "Similarity"],
        ascending=[False, False]
    ).head(top_k)

    return results[
        [
            "Title",
            "Ingredients_Text",
            "Instructions",
            "Matched_Ingredients",
            "Missing_Ingredients",
            "Similarity"
        ]
    ]


print("ระบบค้นหาสูตรอาหารพร้อมใช้งาน")


ระบบค้นหาสูตรอาหารพร้อมใช้งาน


In [98]:
# ทดลองตรวจสอบการแยกวัตถุดิบ
test_results = search_recipes("shrimp, garlic", top_k=3)

for _, row in test_results.iterrows():
    print("เมนู:", row["Title"])
    print("วัตถุดิบที่มี:", row["Matched_Ingredients"])
    print("วัตถุดิบที่ขาด:", row["Missing_Ingredients"])
    print("-" * 50)

เมนู: Grilled Shrimp with Chile, Cilantro, and Lime
วัตถุดิบที่มี: ['shrimp', 'garlic']
วัตถุดิบที่ขาด: ['6 tablespoons fish sauce (such as nuoc nam or nam pla), divided', '4 tablespoons vegetable oil, divided, plus more for grill', '2 teaspoons turmeric', '30 cilantro sprigs, chopped', '10 long red chiles (such as Holland or Anaheim), stemmed', '2 1" pieces ginger, peeled', '1/4 cup fresh lime juice', '2 tablespoons palm sugar or (packed) light brown sugar']
--------------------------------------------------
เมนู: Quick-Pickled Shrimp
วัตถุดิบที่มี: ['shrimp', 'garlic']
วัตถุดิบที่ขาด: ['Kosher salt', '1/2 medium fennel bulb, thinly sliced', '1/2 medium onion, thinly sliced', '1/2 Fresno chile or red jalapeño, thinly sliced, seeded', '1/2 cup fresh lemon juice', '1/4 cup apple cider vinegar', '1/4 cup extra-virgin olive oil', '2 tablespoons fennel fronds', 'Freshly ground black pepper', '1/2 loaf ciabatta or country-style bread, sliced 1/2" thick, toasted', 'Mayonnaise (for serving; o

In [99]:
# 17) ทดลองค้นหาสูตรอาหารก่อนเชื่อมต่อ Gemini
search_test = search_recipes("chicken, egg, garlic", top_k=5)
display(search_test)


,Title,Ingredients_Text,Instructions,Matched_Ingredients,Missing_Ingredients,Similarity
2119,Egg Shop Fried Chicken,"6 cups water, 1/4 cup kosher salt, 1/4 cup pac...","To make the brine, in a large saucepan, combin...","[chicken, egg, garlic]","[6 cups water, 1/4 cup kosher salt, 1/4 cup pa...",0.688694
8845,Chicken in Chile Sauce (Ají de Gallina),"8 quail eggs (optional), 4 (1/2-inch-thick) sl...",Preheat oven to 400°F with rack in middle.\nIf...,"[chicken, egg, garlic]",[4 (1/2-inch-thick) slices firm white sandwich...,0.683862
10764,Ricotta-and-Herb-Stuffed Chicken,"4 garlic cloves, minced, 3 tablespoons extra-v...",Preheat oven to 400°F with racks in middle and...,"[chicken, egg, garlic]","[3 tablespoons extra-virgin olive oil, 4 cups ...",0.676667
9997,Spiced Matzo-Stuffed Chicken Breasts,"1 large onion, chopped, 1 red bell pepper, cho...",Preheat oven to 425°F with racks in upper and ...,"[chicken, egg, garlic]","[1 large onion, chopped, 1 red bell pepper, ch...",0.647544
1324,Make-Ahead Crispy Chicken Cutlets,"2 large egg yolks, 1/2 cup mayonnaise, 2 tsp. ...","Whisk egg yolks, mayonnaise, and salt in a lar...","[chicken, egg, garlic]","[1/2 cup mayonnaise, 2 tsp. kosher salt, 3/4 c...",0.642432


In [100]:

# 18) เชื่อมต่อ Gemini ผ่าน Colab Secrets
from google import genai
from google.genai import types

gemini_key = userdata.get("GEMINI_API_KEY")

if not gemini_key:
    raise ValueError(
        "ไม่พบ GEMINI_API_KEY ใน Colab Secrets กรุณาเพิ่ม Secret ชื่อนี้ก่อนค่ะ"
    )

# เพิ่มเวลารอการเชื่อมต่อเป็น 120 วินาที
client = genai.Client(
    api_key=gemini_key,
    http_options=types.HttpOptions(timeout=120000)
)

print("เชื่อมต่อ Gemini สำเร็จ")


เชื่อมต่อ Gemini สำเร็จ


In [174]:
# 19) AI Recipe Chatbot - Batch Translation System

import re
import json
import time
import random
import hashlib

# ==================================================
# 1. ตั้งค่าระบบแปลภาษา
# ==================================================

TRANSLATION_MODEL = "gemini-3.5-flash-lite"

# Cache เก็บเฉพาะผลแปลที่ผ่านการตรวจสอบแล้ว
recipe_translation_cache = {}


# ==================================================
# 2. เตรียมรายการวัตถุดิบ
# ==================================================

def prepare_ingredient_list(items):

    if items is None:
        return []

    if isinstance(items, list):
        return [
            str(x).strip()
            for x in items
            if str(x).strip()
        ]

    if isinstance(items, str):

        text = items.strip()

        if not text:
            return []

        try:
            parsed = json.loads(text)

            if isinstance(parsed, list):
                return [
                    str(x).strip()
                    for x in parsed
                    if str(x).strip()
                ]

        except Exception:
            pass

        return [
            x.strip()
            for x in re.split(r"\n|;", text)
            if x.strip()
        ]

    return [str(items).strip()]


# ==================================================
# 3. อ่าน JSON จาก Gemini
# ==================================================

def extract_json_object(text):

    if not text:
        raise ValueError("Gemini ไม่ได้ส่งข้อความกลับมา")

    text = str(text).strip()

    text = re.sub(
        r"^```(?:json)?\s*",
        "",
        text
    )

    text = re.sub(
        r"\s*```$",
        "",
        text
    )

    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1:
        raise ValueError("ไม่พบ JSON Object")

    result = json.loads(text[start:end + 1])

    if not isinstance(result, dict):
        raise ValueError("ผลลัพธ์ไม่ใช่ JSON Object")

    return result


# ==================================================
# 4. เรียก Gemini พร้อมระบบลองใหม่เมื่อเกิดข้อผิดพลาด
# ==================================================

def call_gemini_json(prompt, max_attempts=5):

    retryable_errors = [
        "503",
        "UNAVAILABLE",
        "429",
        "RESOURCE_EXHAUSTED",
        "500",
        "502",
        "504",
        "INTERNAL",
        "TIMEOUT"
    ]

    for attempt in range(max_attempts):

        try:

            response = client.models.generate_content(
                model=TRANSLATION_MODEL,
                contents=prompt,
                config={
                    "response_mime_type": "application/json"
                }
            )

            return extract_json_object(response.text)

        except Exception as e:

            error_text = str(e).upper()

            is_retryable = any(
                error in error_text
                for error in retryable_errors
            )

            if not is_retryable or attempt == max_attempts - 1:
                raise

            wait_time = min(
                2 ** (attempt + 1),
                30
            ) + random.uniform(0, 1)

            print(
                f"Gemini ยังไม่พร้อมให้บริการ "
                f"กำลังลองใหม่ครั้งที่ {attempt + 2}/{max_attempts} "
                f"ในอีก {wait_time:.1f} วินาที"
            )

            time.sleep(wait_time)

    raise RuntimeError("Gemini ไม่สามารถตอบกลับได้")


# ==================================================
# 5. เตรียมวิธีทำจาก Dataset
# ==================================================

def split_instructions(instructions):

    if instructions is None:
        return []

    text = str(instructions).strip()

    if not text:
        return []

    lines = [
        line.strip()
        for line in re.split(r"\n+", text)
        if line.strip()
    ]

    cleaned = []

    for line in lines:

        line = re.sub(
            r"^\s*\d+\s*[.)]\s*",
            "",
            line
        ).strip()

        if not line:
            continue

        # ตัดข้อมูลแนะนำไวน์ที่ไม่ใช่ขั้นตอนทำอาหาร
        if re.search(
            r"\b(wine pairing|serve with wine|wine recommendation)\b",
            line,
            re.IGNORECASE
        ):
            continue

        if re.search(
            r"\b(sauvignon blanc|chardonnay|merlot|cabernet sauvignon)\b",
            line,
            re.IGNORECASE
        ):
            continue

        cleaned.append(line)

    return cleaned


# ==================================================
# 6. จัดรูปแบบวิธีทำ
# ==================================================

def format_translated_instructions(items):

    if isinstance(items, str):
        items = [
            x.strip()
            for x in items.splitlines()
            if x.strip()
        ]

    if not isinstance(items, list):
        return ""

    formatted = []

    for item in items:

        text = str(item).strip()

        text = re.sub(
            r"^\s*\d+\s*[.)]\s*",
            "",
            text
        ).strip()

        if text:
            formatted.append(text)

    return "\n\n".join(
        f"{i}. {step}"
        for i, step in enumerate(formatted, start=1)
    )


# ==================================================
# 7. ตรวจสอบความครบถ้วนของคำแปล
# ==================================================

def validate_translation(original, translated):

    if not isinstance(translated, dict):
        return False

    required_fields = [
        "ingredients_you_have",
        "ingredients_missing",
        "cooking_instructions"
    ]

    for field in required_fields:

        if field not in translated:
            return False

        if not isinstance(translated[field], list):
            return False

    # จำนวนวัตถุดิบต้องเท่ากับต้นฉบับ
    if len(translated["ingredients_you_have"]) != len(
        original["matched"]
    ):
        return False

    if len(translated["ingredients_missing"]) != len(
        original["missing"]
    ):
        return False

    # จำนวนขั้นตอนต้องเท่ากับต้นฉบับ
    if len(translated["cooking_instructions"]) != len(
        original["instructions"]
    ):
        return False

    # ห้ามมีข้อความว่าง
    all_items = (
        translated["ingredients_you_have"]
        + translated["ingredients_missing"]
        + translated["cooking_instructions"]
    )

    if any(
        not isinstance(x, str) or not x.strip()
        for x in all_items
    ):
        return False

    return True


# ==================================================
# 8. แปลวัตถุดิบและวิธีทำหลายเมนูพร้อมกัน
# ==================================================

def translate_all_recipes(recipes):

    if not recipes:
        return []

    # สร้าง Cache Key จากข้อมูลต้นฉบับ
    cache_data = json.dumps(
        recipes,
        ensure_ascii=False,
        sort_keys=True
    )

    cache_key = hashlib.sha256(
        cache_data.encode("utf-8")
    ).hexdigest()

    # ใช้ Cache เฉพาะผลลัพธ์ที่ผ่านการตรวจสอบ
    if cache_key in recipe_translation_cache:
        print("ใช้ผลแปลจาก Cache")
        return recipe_translation_cache[cache_key]

    # เตรียมข้อมูลส่ง Gemini
    recipe_data = []

    for index, recipe in enumerate(recipes):

        recipe_data.append({
            "recipe_number": index + 1,
            "ingredients_you_have": recipe["matched"],
            "ingredients_missing": recipe["missing"],
            "cooking_instructions": recipe["instructions"]
        })

    prompt = f"""
คุณเป็นนักแปลสูตรอาหารมืออาชีพ
มีความรู้ด้านวัตถุดิบและการทำอาหาร

หน้าที่ของคุณคือแปลข้อมูลสูตรอาหารทั้งหมดเป็นภาษาไทย

ข้อกำหนดสำคัญ:

1. แปลวัตถุดิบทุกชนิดเป็นภาษาไทยที่เข้าใจง่าย
2. แปลวิธีทำทุกขั้นตอนให้ครบถ้วน
3. เรียงลำดับขั้นตอนตามต้นฉบับ
4. ห้ามตัดหรือรวมขั้นตอนเข้าด้วยกัน
5. ห้ามเพิ่มวัตถุดิบหรือขั้นตอนที่ไม่มีในต้นฉบับ
6. รักษาปริมาณ จำนวน เวลา อุณหภูมิ และหน่วยวัดให้ถูกต้อง
7. ใช้ศัพท์การทำอาหารที่ถูกต้องและเป็นธรรมชาติ
8. ชื่อเฉพาะหรือศัพท์เทคนิคที่ไม่มีคำแปลเหมาะสมสามารถคงภาษาอังกฤษได้
9. ห้ามละเว้นข้อความในต้นฉบับ
10. ห้ามแต่งข้อมูลหรือเดาความหมาย
11. ต้องส่งผลลัพธ์กลับครบทุกเมนู
12. จำนวนรายการวัตถุดิบและขั้นตอนต้องเท่ากับต้นฉบับทุกเมนู

ตอบเป็น JSON Object เท่านั้น ตามโครงสร้างนี้:

{{
  "recipes": [
    {{
      "ingredients_you_have": ["คำแปลวัตถุดิบ"],
      "ingredients_missing": ["คำแปลวัตถุดิบ"],
      "cooking_instructions": [
        "คำแปลขั้นตอนที่ 1",
        "คำแปลขั้นตอนที่ 2"
      ]
    }}
  ]
}}

ข้อมูลสูตรอาหาร:

{json.dumps(recipe_data, ensure_ascii=False)}
"""

    translated_recipes = None

    # ลองใหม่หาก JSON หรือข้อมูลที่ได้รับไม่ครบ
    for validation_attempt in range(2):

        try:

            result = call_gemini_json(prompt)

            translated_recipes = result.get("recipes")

            if not isinstance(translated_recipes, list):
                raise ValueError("ไม่พบรายการสูตรอาหาร")

            if len(translated_recipes) != len(recipes):
                raise ValueError("จำนวนเมนูที่แปลไม่ครบ")

            all_valid = all(
                validate_translation(original, translated)
                for original, translated in zip(
                    recipes,
                    translated_recipes
                )
            )

            if not all_valid:
                raise ValueError(
                    "จำนวนวัตถุดิบหรือขั้นตอนที่แปลไม่ตรงกับต้นฉบับ"
                )

            break

        except Exception as e:

            print(
                f"การแปลหรือการตรวจสอบไม่ผ่าน "
                f"รอบที่ {validation_attempt + 1}/2: {e}"
            )

            if validation_attempt == 0:
                print("กำลังลองแปลใหม่อีกครั้ง")

            translated_recipes = None

    # ==================================================
    # 9. เตรียมผลลัพธ์
    # ==================================================

    final_results = []

    if translated_recipes is not None:

        for translated in translated_recipes:

            final_results.append({

                "matched": [
                    str(x).strip()
                    for x in translated["ingredients_you_have"]
                ],

                "missing": [
                    str(x).strip()
                    for x in translated["ingredients_missing"]
                ],

                "instructions": format_translated_instructions(
                    translated["cooking_instructions"]
                ),

                "translation_status": "success"
            })

        # บันทึก Cache เมื่อผ่านการตรวจสอบครบทุกเมนู
        recipe_translation_cache[cache_key] = final_results

        print("แปลสูตรอาหารและตรวจสอบความครบถ้วนเรียบร้อย")

        return final_results

    # ==================================================
    # 10. หากแปลไม่สำเร็จ ให้ใช้ต้นฉบับแทน
    # ==================================================

    print(
        "แจ้งเตือน: ไม่สามารถแปลข้อมูลได้ครบถ้วน "
        "ระบบจะแสดงข้อมูลต้นฉบับแทนเพื่อป้องกันข้อมูลสูญหาย"
    )

    for recipe in recipes:

        final_results.append({

            "matched": recipe["matched"],

            "missing": recipe["missing"],

            "instructions": format_translated_instructions(
                recipe["instructions"]
            ),

            "translation_status": "fallback"
        })

    # ไม่บันทึกผลลัพธ์ที่แปลไม่สำเร็จลง Cache
    return final_results


# ==================================================
# 11. จัดรูปแบบรายการวัตถุดิบ
# ==================================================

def format_ingredient_section(title, items):

    if not items:
        return f"{title}\n- ไม่มีข้อมูล"

    return title + "\n" + "\n".join(
        f"- {item}"
        for item in items
    )


# ==================================================
# 12. ฟังก์ชันหลักของ Chatbot
# ==================================================

def ask_recipe_chatbot(user_ingredients):

    # ตรวจสอบข้อมูลที่ผู้ใช้ป้อน
    if user_ingredients is None or not str(user_ingredients).strip():
        return "กรุณาระบุวัตถุดิบที่ต้องการใช้ก่อนค่ะ"

    # ค้นหาเมนูจาก Dataset
    results = search_recipes(
        user_ingredients,
        top_k=3
    )

    if results is None or results.empty:

        return (
            "ไม่พบเมนูอาหารจากวัตถุดิบที่ระบุค่ะ\n"
            "กรุณาลองเปลี่ยนหรือเพิ่มวัตถุดิบอีกครั้ง"
        )

    # เตรียมข้อมูลทั้ง 3 เมนู
    recipes = []
    titles = []

    for _, recipe in results.iterrows():

        titles.append(
            str(recipe.get("Title", "ไม่ทราบชื่อเมนู"))
        )

        recipes.append({

            "matched": prepare_ingredient_list(
                recipe.get("Matched_Ingredients", [])
            ),

            "missing": prepare_ingredient_list(
                recipe.get("Missing_Ingredients", [])
            ),

            "instructions": split_instructions(
                recipe.get("Instructions", "")
            )
        })

    # แปลทุกเมนูพร้อมกัน
    translated_results = translate_all_recipes(recipes)

    # สร้างผลลัพธ์
    output = [
        "🍳 เมนูอาหารที่แนะนำจากวัตถุดิบของคุณ",
        "=" * 45
    ]

    for index, translated in enumerate(translated_results):

        output.append(
            f"\n🍽️ เมนูที่ {index + 1}: {titles[index]}"
        )

        output.append("-" * 40)

        if translated["translation_status"] == "fallback":
            output.append(
                "⚠️ หมายเหตุ: ระบบแปลภาษาไม่สำเร็จ "
                "จึงแสดงข้อมูลต้นฉบับเพื่อป้องกันข้อมูลผิดพลาด"
            )

        output.append(
            format_ingredient_section(
                "🥬 วัตถุดิบที่คุณมี",
                translated["matched"]
            )
        )

        output.append("")

        output.append(
            format_ingredient_section(
                "🛒 วัตถุดิบที่ยังขาด",
                translated["missing"]
            )
        )

        output.append("")

        output.append("👩‍🍳 วิธีทำ")

        output.append(
            translated["instructions"]
            if translated["instructions"]
            else "ไม่มีข้อมูลวิธีทำ"
        )

    return "\n\n".join(output)


print("ระบบ AI Recipe Chatbot พร้อมใช้งาน")

ระบบ AI Recipe Chatbot พร้อมใช้งาน


In [175]:
# 20) ทดลองถาม Chatbot
import time

start_time = time.time()

answer = ask_recipe_chatbot("chicken, egg, garlic")

print(answer)

print(f"\nใช้เวลาทั้งหมด {time.time() - start_time:.2f} วินาที")

แปลสูตรอาหารและตรวจสอบความครบถ้วนเรียบร้อย
🍳 เมนูอาหารที่แนะนำจากวัตถุดิบของคุณ



🍽️ เมนูที่ 1: Egg Shop Fried Chicken

----------------------------------------

🥬 วัตถุดิบที่คุณมี
- ไก่
- ไข่
- กระเทียม



🛒 วัตถุดิบที่ยังขาด
- น้ำ 6 ถ้วย
- เกลือโคเชอร์ 1/4 ถ้วย
- น้ำตาลทรายแดงอัดแน่น 1/4 ถ้วย
- เกล็ดพริกแดง 2 ช้อนโต๊ะ
- บัตเติลมิลล์ 2 ถ้วย
- เกลือขึ้นฉ่าย 1 ช้อนชา
- พริกคาเยน 1/2 ช้อนชา
- ปาปริก้ารมควัน 1/2 ช้อนชา
- น้ำมันคาโนลา 3 ถึง 4 ถ้วย สำหรับทอด (ลึกอย่างน้อย 2 นิ้ว)
- แป้งอเนกประสงค์ 2 ถ้วย
- พริกไทยดำบดใหม่ 2 ช้อนโต๊ะ
- น้ำผึ้งป่าและเกลือทะเลหยาบ สำหรับเสิร์ฟ



👩‍🍳 วิธีทำ

1. ในการทำน้ำเกลือสำหรับหมัก ให้ผสมน้ำ เกลือ น้ำตาลทรายแดง กระเทียม และเกล็ดพริกในหม้อขนาดใหญ่ ตั้งไฟปานกลางจนเดือด พักไว้ให้เย็นสนิท

2. ใส่ไก่ลงในน้ำเกลือและแช่ทิ้งไว้ 24 ชั่วโมงในตู้เย็น หรืออย่างน้อยข้ามคืน กรองไก่ขึ้นจากน้ำเกลือ

3. อย่างน้อย 1 ชั่วโมงก่อนที่คุณจะพร้อมทอดไก่ (และล่วงหน้าได้ถึง 24 ชั่วโมง) ให้ทำส่วนผสมหมักในชามขนาดใหญ่ ตีบัตเติลมิลล์ เกลือขึ้นฉ่าย พริกคาเยน ปาปริก้า และไข่ให้เข้ากัน ใ

In [179]:

# 21) สร้างหน้าเว็บ AI Recipe Chatbot รองรับภาษาไทยและอังกฤษ

import gradio as gr

# 1. พจนานุกรมแปลงวัตถุดิบภาษาไทยเป็นภาษาอังกฤษ
ingredient_translation = {
    "ไก่": "chicken",
    "เนื้อไก่": "chicken",
    "ไข่": "egg",
    "ไข่ไก่": "egg",
    "กระเทียม": "garlic",
    "ข้าว": "rice",
    "มะเขือเทศ": "tomato",
    "หอมหัวใหญ่": "onion",
    "หัวหอม": "onion",
    "มันฝรั่ง": "potato",
    "แครอท": "carrot",
    "หมู": "pork",
    "เนื้อหมู": "pork",
    "เนื้อวัว": "beef",
    "ปลา": "fish",
    "กุ้ง": "shrimp",
    "นม": "milk",
    "ชีส": "cheese",
    "เนย": "butter",
    "เกลือ": "salt",
    "น้ำตาล": "sugar",
    "พริก": "chili",
    "ขิง": "ginger",
    "ต้นหอม": "green onion",
    "ผักชี": "cilantro",
    "แตงกวา": "cucumber",
    "เห็ด": "mushroom"
}


# 2. ฟังก์ชันแปลงวัตถุดิบ
def normalize_ingredients(user_input):
    items = (
        user_input
        .replace("，", ",")
        .replace(";", ",")
        .replace("\n", ",")
        .split(",")
    )

    translated_items = []

    for item in items:
        item = item.strip().lower()

        if not item:
            continue

        translated = ingredient_translation.get(item, item)
        translated_items.append(translated)

    return ", ".join(translated_items)


# 3. ธีมเว็บไซต์
APP_THEME = gr.themes.Soft(
    primary_hue="orange",
    secondary_hue="orange",
    neutral_hue="stone"
)


# 4. CSS ตกแต่งเว็บไซต์
CUSTOM_CSS = """
.gradio-container {
    max-width: 1150px !important;
    margin: auto !important;
    background: #f7f3ed !important;
    color: #38271d !important;
    font-family: Arial, sans-serif !important;
}

body {
    background: #f7f3ed !important;
}

#hero {
    position: relative;
    overflow: hidden;
    min-height: 300px;
    border-radius: 24px;
    margin: 18px 0 24px;
    padding: 55px 40px;
    display: flex;
    align-items: center;
    background:
      linear-gradient(90deg, rgba(28,19,14,.96), rgba(28,19,14,.75), rgba(28,19,14,.12)),
      url("https://images.unsplash.com/photo-1546069901-ba9599a7e63c?w=1400") center/cover;
}

#hero-content {
    max-width: 620px;
}

#hero h1 {
    color: #fff8ee !important;
    font-size: 42px !important;
    font-weight: 800 !important;
    margin-bottom: 12px !important;
}

#hero p {
    color: #f0dfcc !important;
    font-size: 17px !important;
    line-height: 1.8 !important;
}

.hero-tag {
    color: #e6b77e;
    font-size: 12px;
    letter-spacing: 3px;
    font-weight: bold;
    margin-bottom: 12px;
}

.section-title {
    color: #493123 !important;
    font-size: 23px !important;
    font-weight: 750 !important;
}

.section-subtitle {
    color: #8b7768 !important;
    font-size: 14px !important;
}

#search-panel, #result-panel {
    background: #fffdf9 !important;
    border: 1px solid #e8ddd0 !important;
    border-radius: 20px !important;
    padding: 24px !important;
    box-shadow: 0 5px 20px rgba(70,45,25,.06) !important;
}

#result-panel {
    min-height: 220px;
}

label {
    color: #493123 !important;
    font-weight: 650 !important;
}

textarea, input {
    background: #fffaf3 !important;
    color: #38271d !important;
    border: 1px solid #d9c6b3 !important;
    border-radius: 12px !important;
}

textarea::placeholder {
    color: #aa9a8b !important;
}

button.primary {
    background: #875b3e !important;
    color: white !important;
    border: none !important;
    border-radius: 12px !important;
    font-weight: bold !important;
    min-height: 46px !important;
}

button.primary:hover {
    background: #68432d !important;
}

button.secondary {
    border-radius: 12px !important;
}

#result-panel .prose,
#result-panel .markdown {
    color: #493123 !important;
    line-height: 1.8 !important;
}

#result-panel h1,
#result-panel h2,
#result-panel h3 {
    color: #70482f !important;
}

#result-panel strong {
    color: #70482f !important;
}

.gradio-container .wrap {
    border-radius: 12px;
}

footer {
    display: none !important;
}

@media (max-width: 650px) {
    #hero {
        padding: 35px 22px;
        min-height: 250px;
    }

    #hero h1 {
        font-size: 30px !important;
    }

    #search-panel, #result-panel {
        padding: 16px !important;
    }
}
"""


# 5. ฟังก์ชันเรียกใช้งาน Chatbot
def chat_with_recipe(user_ingredients):

    if not user_ingredients or not user_ingredients.strip():
        return "กรุณากรอกวัตถุดิบก่อนนะคะ 😊"

    try:
        # แปลงภาษาไทยเป็นภาษาอังกฤษก่อนค้นหา
        translated_ingredients = normalize_ingredients(
            user_ingredients
        )

        print("วัตถุดิบที่ผู้ใช้กรอก:", user_ingredients)
        print("วัตถุดิบที่ส่งไปค้นหา:", translated_ingredients)

        # เรียกใช้ระบบ RAG และ Gemini เดิม
        answer = ask_recipe_chatbot(translated_ingredients)

        return answer

    except Exception as error:
        print("Chatbot error:", repr(error))
        return "ขออภัยค่ะ ระบบเกิดข้อผิดพลาด กรุณาลองใหม่อีกครั้ง"


# 6. สร้างหน้าเว็บไซต์
with gr.Blocks(title="AI Recipe Chatbot") as demo:

    # ส่วนหัวเว็บไซต์
    gr.HTML("""
    <div id="hero">
        <div id="hero-content">
            <div class="hero-tag">
                YOUR PERSONAL AI COOKING ASSISTANT
            </div>

            <h1>AI Recipe Chatbot</h1>

            <p>
                เปลี่ยนวัตถุดิบที่คุณมีให้กลายเป็นมื้ออาหารแสนอร่อย
                ค้นหาเมนูที่เหมาะกับคุณ พร้อมคำแนะนำวิธีทำจาก AI
            </p>
        </div>
    </div>
    """)

    # ส่วนค้นหาวัตถุดิบ
    with gr.Column(elem_id="search-panel"):

        gr.Markdown(
            "## 🥘 วันนี้มีวัตถุดิบอะไรในครัวบ้าง?",
            elem_classes=["section-title"]
        )

        gr.Markdown(
            "กรอกวัตถุดิบที่คุณมี แล้วให้ AI ช่วยค้นหาเมนูอาหารที่เหมาะสม",
            elem_classes=["section-subtitle"]
        )

        ingredients_input = gr.Textbox(
            label="วัตถุดิบของคุณ",
            placeholder="เช่น ไก่, ไข่, กระเทียม หรือ chicken, egg, garlic",
            lines=2
        )

        gr.Markdown(
            "<span style='color:#8b7768'>"
            "💡 สามารถกรอกวัตถุดิบได้ทั้งภาษาไทยและภาษาอังกฤษ "
            "โดยคั่นแต่ละรายการด้วยเครื่องหมายจุลภาค (,)"
            "</span>"
        )

        with gr.Row():

            submit_btn = gr.Button(
                "ค้นหาเมนูอาหาร ✨",
                variant="primary",
                scale=3
            )

            clear_btn = gr.ClearButton(
                [ingredients_input],
                value="ล้างข้อมูล",
                scale=1
            )

    # ตัวอย่างวัตถุดิบ
    with gr.Accordion("🍅 ลองเลือกชุดวัตถุดิบตัวอย่าง", open=False):

        gr.Examples(
            examples=[
                ["ไก่, ไข่, กระเทียม"],
                ["ข้าว, ไข่, หอมหัวใหญ่"],
                ["มะเขือเทศ, มันฝรั่ง, หอมหัวใหญ่"],
                ["chicken, egg, garlic"]
            ],
            inputs=ingredients_input
        )

    # ส่วนแสดงผลลัพธ์
    with gr.Column(elem_id="result-panel"):

        gr.Markdown(
            "## ✨ เมนูอาหารที่แนะนำ",
            elem_classes=["section-title"]
        )

        gr.Markdown(
            "ผลการค้นหาและคำแนะนำจาก AI จะแสดงที่นี่",
            elem_classes=["section-subtitle"]
        )

        answer_output = gr.Markdown(
            "🍽️ **พร้อมแล้วหรือยัง?** "
            "กรอกวัตถุดิบด้านบน แล้วกดค้นหาเมนูอาหารได้เลยค่ะ"
        )

    # เชื่อมปุ่มค้นหากับ Chatbot
    submit_btn.click(
        fn=chat_with_recipe,
        inputs=ingredients_input,
        outputs=answer_output
    )

    ingredients_input.submit(
        fn=chat_with_recipe,
        inputs=ingredients_input,
        outputs=answer_output
    )


print("สร้างหน้าเว็บ AI Recipe Chatbot รองรับภาษาไทยและอังกฤษเรียบร้อยแล้ว")


สร้างหน้าเว็บ AI Recipe Chatbot รองรับภาษาไทยและอังกฤษเรียบร้อยแล้ว


In [180]:

# 22) เปิดหน้าเว็บ AI Recipe Chatbot

demo.launch(
    share=True,
    debug=False,
    theme=APP_THEME,
    css=CUSTOM_CSS
)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cb0f864e828e9e1c23.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
